In [1]:
import numpy as np
from scipy import linalg


In [2]:
# Line reactances in per unit. Resistance is neglected in the DC power-flow model.
# R = 0.0 pu for all lines. therefore, only the reactance (X) is given. The susceptance (B) is calculated as B = 1 / X.
# B = X/R^2 + X^2 = 1 / X (since R = 0)


X = np.array([0.2, 0.4, 0.25])  # Reactance of lines 12, 13, and 32 in per unit
B = 1 / X  # Calculate susceptance for each lin
print("Line reactances (X):", X
      )
print("Line susceptances (B):", B)



Line reactances (X): [0.2  0.4  0.25]
Line susceptances (B): [5.  2.5 4. ]


In [4]:
# Direct calculation using theta_1 as the reference angle.
# P_ij = B_ij * (theta_i - theta_j)
theta_direct = np.zeros(3)

P12 = 0.62
P13 = 0.06
P32_measured = 0.37

B12, B13, B32 = B
theta_direct[1] = -P12 / B12
theta_direct[2] = -P13 / B13
P32_from_direct_angles = B32 * (theta_direct[2] - theta_direct[1])

print("Direct angle estimate with theta_1 = 0:")
print(f"theta_1 = {theta_direct[0]: .3f} rad")
print(f"theta_2 = {theta_direct[1]: .4f} rad")
print(f"theta_3 = {theta_direct[2]: .4f} rad")
print()
print(f"Measured P32 = {P32_measured:.3f} pu")
print(f"P32 implied by direct angles = {P32_from_direct_angles:.3f} pu")


Direct angle estimate with theta_1 = 0:
theta_1 =  0.000 rad
theta_2 = -0.1240 rad
theta_3 = -0.0240 rad

Measured P32 = 0.370 pu
P32 implied by direct angles = 0.400 pu


In [ ]:
# Least-squares state estimate using all three line-flow measurements.
# Unknown state vector is x = [theta_2, theta_3] because theta_1 is fixed at 0.
z = np.array([P12, P13, P32_measured])
# H is the Jacobian matrix of the measurement equations
H = np.array([
    [-B12, 0.0],       # P12 = B12 * (theta_1 - theta_2) = -B12 * theta_2
    [0.0, -B13],       # P13 = B13 * (theta_1 - theta_3) = -B13 * theta_3
    [-B32, B32],       # P32 = B32 * (theta_3 - theta_2)
])

sigma = 0.01 # Standard deviation of measurement noise in per unit
R = (sigma ** 2) * np.eye(len(z))
W = linalg.inv(R)

normal_matrix = H.T @ W @ H  # math equation is H^T * W * H
normal_rhs = H.T @ W @ z # math equation is H^T * W * z
theta_unknown = linalg.solve(normal_matrix, normal_rhs, assume_a="pos")

theta_ls = np.array([0.0, theta_unknown[0], theta_unknown[1]])
z_hat = H @ theta_unknown
residual = z - z_hat

print("Weighted least-squares angle estimate with theta_1 = 0:")
print(f"theta_1 = {theta_ls[0]: .6f} rad")
print(f"theta_2 = {theta_ls[1]: .6f} rad")
print(f"theta_3 = {theta_ls[2]: .6f} rad")
print()
print("Measurement check [P12, P13, P32] in pu:")
print(f"measured  = {z}")
print(f"estimated = {np.round(z_hat, 6)}")
print(f"residual  = {np.round(residual, 6)}")


Weighted least-squares angle estimate with theta_1 = 0:
theta_1 =  0.000000 rad
theta_2 = -0.122857 rad
theta_3 = -0.028571 rad

Measurement check [P12, P13, P32] in pu:
measured  = [0.62 0.06 0.37]
estimated = [0.614286 0.071429 0.377143]
residual  = [ 0.005714 -0.011429 -0.007143]


In [ ]:
# Same least-squares solution using theta_3 as the reference, matching the lecture note convention.
# Unknown state vector is x = [theta_1, theta_2], with theta_3 = 0.
H_theta3_ref = np.array([
    [B12, -B12],       # P12 = B12 * (theta_1 - theta_2)
    [B13, 0.0],        # P13 = B13 * (theta_1 - theta_3) = B13 * theta_1
    [0.0, -B32],       # P32 = B32 * (theta_3 - theta_2) = -B32 * theta_2
])

theta13_unknown = linalg.solve(
    H_theta3_ref.T @ W @ H_theta3_ref,
    H_theta3_ref.T @ W @ z,
    assume_a="pos", # assume the normal matrix is positive definite for numerical stability
)

theta_theta3_ref = np.array([theta13_unknown[0], theta13_unknown[1], 0.0])

print("Equivalent estimate with theta_3 = 0:")
print(f"theta_1 = {theta_theta3_ref[0]: .6f} rad")
print(f"theta_2 = {theta_theta3_ref[1]: .6f} rad")
print(f"theta_3 = {theta_theta3_ref[2]: .6f} rad")


Equivalent estimate with theta_3 = 0:
theta_1 =  0.028571 rad
theta_2 = -0.094286 rad
theta_3 =  0.000000 rad
